# 04 — Modélisation de l’agressivité / corrosivité

Cible : `Larson_corrosive = 1` si IC ≥ 1. La cible dépend de Cl, SO4 et HCO3 ; cette dépendance doit être discutée comme limite méthodologique.

## 1. Configuration

In [ ]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
DATA_RAW=PROJECT_ROOT/"data"/"raw"; DATA_PROCESSED=PROJECT_ROOT/"data"/"processed"
FIGURES=PROJECT_ROOT/"reports"/"figures"; RESULTS=PROJECT_ROOT/"reports"/"results"; MODELS=PROJECT_ROOT/"models"
for p in [DATA_PROCESSED,FIGURES,RESULTS,MODELS]: p.mkdir(parents=True,exist_ok=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
RANDOM_STATE=42
pd.set_option("display.max_columns",100)

In [ ]:
from src.data_cleaning import load_water_quality,basic_cleaning
from src.indices_chimiques import add_larson_features
from sklearn.model_selection import train_test_split
from src.preprocessing import make_model_pipeline
from src.modeling import classification_models
from src.evaluation import classification_metrics,cross_validation_table,save_confusion_matrix,save_roc_curve
from src.interpretability import permutation_importance_table
import joblib

## 2. Construction des données

In [ ]:
p=DATA_RAW/"water_quality.xlsx"
m=add_larson_features(basic_cleaning(load_water_quality(p,"MHLATHUZE"))); l=add_larson_features(basic_cleaning(load_water_quality(p,"LUVUVU")))
common=set(m.columns)&set(l.columns); features=[c for c in ["pH","EC","TDS","Na","K","Ca","Mg","Cl","HCO3","SO4","NO3"] if c in common]
data=pd.concat([m[features+["Larson_corrosive"]],l[features+["Larson_corrosive"]]],ignore_index=True).replace([np.inf,-np.inf],np.nan).dropna(subset=["Larson_corrosive"])
data["Larson_corrosive"]=data["Larson_corrosive"].astype(int); display(data["Larson_corrosive"].value_counts())

## 3. X et y

In [ ]:
X=data[features]; y=data["Larson_corrosive"]
print(features)

## 4. Train/test

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=RANDOM_STATE)

## 5. Validation croisée

In [ ]:
models=classification_models(RANDOM_STATE); cv_rows=[]; pipes={}
for name,model in models.items():
    pipe=make_model_pipeline(model,X.columns,scale=name in ["Logistic Regression","SVM-RBF"]); pipes[name]=pipe
    cv=cross_validation_table(pipe,X_train,y_train,5,RANDOM_STATE)
    for _,r in cv.iterrows(): cv_rows.append({"model":name,**r.to_dict()})
cv_results=pd.DataFrame(cv_rows); display(cv_results.pivot(index="model",columns="metric",values="mean").round(4))

## 6. Test final

In [ ]:
rows=[]; fitted={}
for name,pipe in pipes.items():
    pipe.fit(X_train,y_train); fitted[name]=pipe; rows.append({"model":name,**classification_metrics(pipe,X_test,y_test)})
test_results=pd.DataFrame(rows).sort_values(["f1","roc_auc"],ascending=False).reset_index(drop=True); display(test_results.round(4))

## 7. Meilleur modèle et figures

In [ ]:
best_name=test_results.iloc[0]["model"]; best=fitted[best_name]
cm=FIGURES/"aggressiveness_confusion_matrix.png"; roc=FIGURES/"aggressiveness_roc_curve.png"
save_confusion_matrix(best,X_test,y_test,f"Larson — {best_name}",cm); save_roc_curve(best,X_test,y_test,f"Larson — {best_name}",roc)
from IPython.display import Image,display
display(Image(filename=str(cm))); display(Image(filename=str(roc)))

## 8. Importance des variables

In [ ]:
imp=permutation_importance_table(best,X_test,y_test); display(imp)
imp.sort_values("importance_mean").plot.barh(x="feature",y="importance_mean",xerr="importance_std",figsize=(7,5),legend=False); plt.tight_layout(); plt.show()

## 9. Sauvegarde

In [ ]:
cv_results.to_csv(RESULTS/"aggressiveness_cv_metrics.csv",index=False); test_results.to_csv(RESULTS/"aggressiveness_test_metrics.csv",index=False); imp.to_csv(RESULTS/"aggressiveness_feature_importance.csv",index=False); joblib.dump(best,MODELS/"aggressiveness_best_model.pkl")

## 10. Discussion obligatoire

Le modèle automatise une classe dérivée de Larson. Il ne constitue pas une mesure indépendante de corrosion. Une validation industrielle nécessiterait des observations de corrosion ou une cible experte indépendante.